In [ ]:
import os
import re
from datetime import datetime
from pathlib import Path

import ee
import geemap 

import gee_export_helpers as gee_helpers


#####################################
#    SET ALL PARAMETERS #############
#####################################

CYVERSE = False
PROJECT_DIR = "projects/ee-tymc5571-goodfire/assets"
GEE_PROJECT = "ee-tymc5571-goodfire"
DRIVE_FOLDER      = "GEE_Exports_welty20102020"
CRS = "EPSG:5070"  # export CRS

# ee.Authenticate()
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/devstorage.read_write',
        'https://www.googleapis.com/auth/drive'
    ]
)
ee.Initialize(project=GEE_PROJECT)

if CYVERSE:
    SERVICE_FILE = "~/data-store/data/iplant/home/tylermcintosh/secrets/tymc5571-utils-project-692f27c034dd.json"  # Path to service account file
    DIR_DERIVED = "~/data-store/data/iplant/home/shared/earthlab/macrosystems/cbi-module/data/derived/"
else:
    SERVICE_FILE = "C:/Users/tymc5571/dev/cbi-module/config/secrets/tymc5571-utils-project-692f27c034dd.json"  # Path to service account file
    DIR_DERIVED = "C:/Users/tymc5571/dev/cbi-module/data/derived/"

    

Path(DIR_DERIVED).mkdir(parents=True, exist_ok=True)


c:\Users\tymc5571\AppData\Local\miniconda3\envs\cbi-module\lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
import importlib
importlib.reload(gee_helpers)


<module 'gee_export_helpers' from 'c:\\Users\\tymc5571\\dev\\a-number-on-good-fire\\code\\gee_export_helpers.py'>

In [4]:
# filter to assets of interest

gee_helpers.list_assets(PROJECT_DIR)

assets = gee_helpers.list_assets(PROJECT_DIR)
filtered_assets = [a for a in assets if ("splitfrg5" in a and (m := re.search(r'(\d{4})$', a)) and 2010 <= int(m.group(1)) <= 2020)]
print(filtered_assets)

['projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2010', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2011', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2012', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2013', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2014', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2015', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2016', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2017', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2018', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2019', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2020']


# Create clean versions

In [18]:
og_im = ee.Image(filtered_assets[0])

og_im.bandNames().getInfo()


def make_gwf_classified(img):
    empty = ee.Image.constant(0).toInt16()

    gwf_class = (
        empty
        .where(img.select("lowerGoodFire").eq(900), 1)
        .where(img.select("highGoodFire").eq(900), 2)
        .where(img.select("unmatchedRegime").eq(900), 3)
        .where(img.select("tooFrequent").eq(900), 4)
        .selfMask()
        .rename("gwf_class")
    )

    gwf_class_granular = (
        empty
        .where(img.select("frg1Good").eq(900), 1)
        .where(img.select("frg2Good").eq(900), 2)
        .where(img.select("frg3Good").eq(900), 3)
        .where(img.select("frg4Good").eq(900), 4)
        .where(img.select("frg5lGood").eq(900), 5)
        .where(img.select("frg5hGood").eq(900), 6)
        .where(img.select("lowerRegimeCbiHigh").eq(900), 7)
        .where(img.select("replaceRegimeCbiLow").eq(900), 8)
        .where(img.select("tooFrequentLow").eq(900), 9)
        .where(img.select("tooFrequentHigh").eq(900), 10)
        .selfMask()
        .rename("gwf_class_granular")
    )

    return ee.Image.cat([gwf_class, gwf_class_granular])

# Example usage
gwf_img = make_gwf_classified(og_im)
gwf_img.bandNames().getInfo()

['gwf_class', 'gwf_class_granular']

In [ ]:
# map = geemap.Map()

# # coarse classes:
# # 1 = lowerGoodFire
# # 2 = highGoodFire
# # 3 = unmatchedRegime
# # 4 = tooFrequent
# coarse_vis = {
#     "min": 1,
#     "max": 4,
#     "palette": [
#         "00FFFF",  # bright cyan
#         "FF00FF",  # bright magenta
#         "FFFF00",  # bright yellow
#         "00FF00",  # bright lime
#     ]
# }

# # granular classes 1-10
# granular_vis = {
#     "min": 1,
#     "max": 10,
#     "palette": [
#         "FF0000",  # 1
#         "00FF00",  # 2
#         "0000FF",  # 3
#         "FFFF00",  # 4
#         "FF00FF",  # 5
#         "00FFFF",  # 6
#         "FFA500",  # 7
#         "7FFF00",  # 8
#         "9400D3",  # 9
#         "FF1493",  # 10
#     ]
# }

# # original in red
# og_vis = {
#     "min": 900,
#     "max": 900,
#     "palette": ["FF0000"]
# }

# map.addLayer(gwf_img.select("gwf_class"), coarse_vis, "GWF Coarse Class")
# map.addLayer(gwf_img.select("gwf_class_granular"), granular_vis, "GWF Granular Class")
# map.addLayer(og_im.select("goodFireAll"), og_vis, "og")

# map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

# Export

In [20]:
for asset in filtered_assets:

    clean_img = make_gwf_classified(ee.Image(asset))

    task = ee.batch.Export.image.toDrive(
        image=clean_img,
        description=f'gwf_export_{asset[-4:]}',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'gwf_classified_welty_{asset[-4:]}',
        #region=west_geom,
        scale=30,
        crs=CRS,
        maxPixels=1e13
    )
    task.start()
    print(f'Started export task for {asset} with task ID: gwf_export_{asset[-4:]}')

Started export task for projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2010 with task ID: gwf_export_2010
Started export task for projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2011 with task ID: gwf_export_2011
Started export task for projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2012 with task ID: gwf_export_2012
Started export task for projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2013 with task ID: gwf_export_2013
Started export task for projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2014 with task ID: gwf_export_2014
Started export task for projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2015 with task ID: gwf_export_2015
Started export task for projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_2016 with task ID: gwf_export_2016
Started export task for projects/ee-tymc5571-goodfire/assets/all_gf_welty_wi

In [21]:
for asset in filtered_assets:

    gee_helpers.download_merge_from_drive(
        description=f'gwf_classified_welty_{asset[-4:]}',
        local_filename=os.path.join(DIR_DERIVED, f'gwf_classified_welty_{asset[-4:]}.tif'),
        drive_folder=DRIVE_FOLDER,
        service_account_file=SERVICE_FILE,
        as_cog=False,
        compress="ZSTD",
        predictor=2, # 3 = best for float32 data (like indices or continuous values), 2 = best for integers
        blocksize=512, # tile size; 256–512 is good default
        bigtiff="IF_SAFER",
        n_workers=6,
        check_existing=True,
        verbose=True
    )


Found 24 file(s). Starting download to C:\Users\tymc5571\AppData\Local\Temp\tmp3v1didg4 ...
Downloaded gwf_classified_welty_2010-0000000000-0000032768.tif
Downloaded gwf_classified_welty_2010-0000000000-0000065536.tif
Downloaded gwf_classified_welty_2010-0000000000-0000163840.tif
Downloaded gwf_classified_welty_2010-0000000000-0000098304.tif
Downloaded gwf_classified_welty_2010-0000000000-0000000000.tif
Downloaded gwf_classified_welty_2010-0000032768-0000000000.tif
Downloaded gwf_classified_welty_2010-0000032768-0000032768.tif
Downloaded gwf_classified_welty_2010-0000032768-0000131072.tif
Downloaded gwf_classified_welty_2010-0000032768-0000098304.tif
Downloaded gwf_classified_welty_2010-0000065536-0000000000.tif
Downloaded gwf_classified_welty_2010-0000065536-0000032768.tif
Downloaded gwf_classified_welty_2010-0000065536-0000065536.tif
Downloaded gwf_classified_welty_2010-0000065536-0000098304.tif
Downloaded gwf_classified_welty_2010-0000065536-0000163840.tif
Downloaded gwf_classified_